# Beehive agents with local Qwen: batched decisions, hidden hive-state vectors

This notebook runs the 50-agent beehive system with the trained **Qwen3.5-0.8B LoRA adapter**. Bees use local observations, personal memory and nearby messages; they never receive the observer's 14-component hive-state vector or researcher scores.

The policy batches up to **16 independent local observations** into one GPU generation call. All observations in a batch come from one frozen hive state. The returned actions are then applied in the recorded order, with a state-vector snapshot after every applied or rejected action. Actions in one batch share a simulated timestamp; later actions can encounter resources or conditions changed by earlier actions. The next batch observes the updated hive.

This is a synthetic engineering simulation, not biological validation. Training labels are hand-written scenario-teacher decisions. See [MODEL_DESIGN.md](MODEL_DESIGN.md) and the attached runtime dataset card.


## 1. Install and load the private Kaggle runtime

Enable a Kaggle GPU and internet access. Attach the private dataset `faithfulchukwunwogor/beehive-qwen35-runtime`; it contains the simulator, local policy and trained adapter. Internet access is used to install Unsloth and load the open-weight Qwen base model. This notebook makes no OpenAI API calls and needs no API key.


In [ ]:
%pip install --upgrade --quiet unsloth unsloth_zoo "transformers>=5.0.0" "trl>=0.27.0" datasets accelerate peft


In [ ]:
from pathlib import Path
import sys

KAGGLE_INPUT = Path("/kaggle/input")
simulator_files = list(KAGGLE_INPUT.rglob("beehive_sim.py"))
if not simulator_files:
    raise FileNotFoundError("Attached runtime dataset does not contain beehive_sim.py")
SIMULATOR_DIR = simulator_files[0].parent
POLICY_FILE = SIMULATOR_DIR / "bee_local_qwen_policy.py"
if not POLICY_FILE.exists():
    raise FileNotFoundError(f"Local policy missing beside simulator: {POLICY_FILE}")
ADAPTER_CANDIDATES = [
    p.parent for p in KAGGLE_INPUT.rglob("adapter_config.json")
    if (p.parent / "adapter_model.safetensors").is_file()
]
if not ADAPTER_CANDIDATES:
    raise FileNotFoundError("Attached runtime dataset does not contain the trained LoRA adapter")
ADAPTER_DIR = ADAPTER_CANDIDATES[0]
if str(SIMULATOR_DIR) not in sys.path:
    sys.path.insert(0, str(SIMULATOR_DIR))
print("Simulator source:", SIMULATOR_DIR)
print("Trained adapter:", ADAPTER_DIR)


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import json
import math

import matplotlib.pyplot as plt
from IPython.display import FileLink, Markdown, display
from beehive_sim import (
    Config, World, Observer, Simulation,
    VECTOR_NAMES, plot_hive, plot_transcript, weighted_research_score,
)
from bee_local_qwen_policy import BeeLocalQwenPolicy


## 2. Adjustable experiment settings

Change `N_BEES`, duration, rounds, action cap, and `LLM_BATCH_SIZE` below. The default is 50 adults, six simulated hours, and 15-minute rounds. At 50 surviving adults that is 1,200 actions. The local model batches up to 16 decisions; set the size to 1 for sequential decisions.

Batched decisions use the same frozen observation state for all bees in a batch, then the simulator applies the outputs in order and snapshots after each action. `BeeLocalQwenPolicy` automatically splits a batch if it runs out of GPU memory.


In [ ]:
N_BEES = 50                 # Includes ONE queen. Must be >= 2.
DURATION_HOURS = 6.0
ROUND_MINUTES = 15.0
MAX_ACTIONS = None          # Example: 32 for a two-batch smoke run.
SEED = 42
MODEL = "unsloth/Qwen3.5-0.8B + trained LoRA adapter"
MAX_NEW_TOKENS = 160
LLM_BATCH_SIZE = 16         # One padded GPU generation call per batch.
START_TIMESTAMP = "2026-01-01T06:00:00+00:00"

# All calibration anchors are fixed before the experiment.
RESERVE_HOURS = 24.0
OPTIMAL_TEMPERATURE_C = 35.0
OPTIMAL_HUMIDITY_PCT = 60.0

action_cap = (MAX_ACTIONS if MAX_ACTIONS is not None
              else N_BEES * math.ceil(DURATION_HOURS * 60 / ROUND_MINUTES))
config = Config(
    n_bees=N_BEES, duration_hours=DURATION_HOURS,
    round_minutes=ROUND_MINUTES, max_actions=action_cap,
    seed=SEED, model=MODEL, start_timestamp=START_TIMESTAMP,
    reserve_hours=RESERVE_HOURS,
    optimal_temperature_c=OPTIMAL_TEMPERATURE_C,
    optimal_humidity_pct=OPTIMAL_HUMIDITY_PCT,
)
print(f"Adults: {N_BEES}; duration: {DURATION_HOURS} h; action cap: {action_cap:,}")
print(f"Local model: {MODEL}; requested GPU batch size: {LLM_BATCH_SIZE}")


## 3. Physical environment and initial scenario

The hive has 19 hexagonal compartments, seven brood locations, an entrance, and a graph linking two finite flower patches and a pond. Bees can move, forage, transfer food, nurse larvae, signal peers, clean, groom, repair comb, regulate temperature and switch worker roles. The queen can lay eggs.

The commented edits below are researcher-controlled scenario interventions. They change physical conditions, not observer scores. Apply them **before** constructing `Simulation` so the saved baseline matches the scenario.

In [ ]:
world = World(config)

# Optional scenario modifications (uncomment/edit BEFORE running):
# world.cells["h:0:0"].temperature_c = 30.0
# world.cells["h:0:0"].debris = 6.0
# world.cells["h:0:0"].structure = 0.45
# world.cells["h:1:0"].stores["honey"] = 0.0
# world.cells["field:meadow"].flowers["nectar"] *= 0.25
# world.cells["field:orchard"].toxin = 0.10
# config.basal_energy_per_hour = 8.0

fig = plot_hive(world)
plt.show()


## 4. Load the local policy and inspect its information boundary

The policy loads the attached LoRA adapter and the open-weight Qwen base model. It receives only local observations. The sample below omits all researcher-only hive-vector values.


In [ ]:
policy = BeeLocalQwenPolicy.from_adapter(
    ADAPTER_DIR,
    max_seq_length=3072,
    max_new_tokens=MAX_NEW_TOKENS,
    batch_size=LLM_BATCH_SIZE,
)
sample_observation = world.observe("bee-001")
print(json.dumps(sample_observation, indent=2)[:5500])
print("\nPolicy label:", policy.label)
print("Adapter directory:", ADAPTER_DIR)
assert all(name not in json.dumps(sample_observation) for name in VECTOR_NAMES)


## 5. Run and return the timestamped vector transcript

For each batch, the simulation advances background time by the batch's slots, captures every living bee's local observation from that shared state, calls batched local inference, applies the returned actions in order, and records an immediate vector snapshot after each action attempt. All events in a decision batch share a timestamp. Action preconditions are checked against the live hive at execution, so conflicting batch outputs can be rejected and are recorded in the audit.

The JSONL/CSV transcript still has one vector row per executed action. The audit includes batch ID, index, batch size, decision timestamp, outcome, local token counts, and state deltas. The batch's observations are intentionally frozen; the next batch sees the updated state.


In [ ]:
run_dir = Path("runs") / (
    f"qwen_local_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}"
)
simulation = Simulation(world, policy, output_dir=run_dir)
transcript = simulation.run(
    progress_every=250, decision_batch_size=LLM_BATCH_SIZE
)

print(f"Recorded {len(transcript):,} post-action vectors in {run_dir}")
print("Vector order:", list(VECTOR_NAMES))
assert len(transcript) == len(simulation.audit)
assert all(len(row["vector"]) == len(VECTOR_NAMES) for row in transcript)
assert all(0.0 <= value <= 1.0 for row in transcript for value in row["vector"])
print(json.dumps({
    "first_two": transcript[:2],
    "last": transcript[-1:] if transcript else [],
}, indent=2))


## 6. Exported vector script and audit

`transcript` holds every timestamp/vector pair in memory. The JSONL file is the long, line-by-line vector script; CSV has the attribute names. Metadata records fixed normalization anchors, configuration, initial/final raw state, prompt, schema, decision batch size and engine hash. These researcher artifacts are never fed back to agents.

Batched audit rows identify the frozen observation batch and each action's execution order. The local model records prompt/completion token counts for compute accounting; there is no API billing or response ID.


In [ ]:
for filename in ("vectors.jsonl", "vectors.csv", "actions.jsonl", "metadata.json"):
    display(FileLink(str(run_dir / filename)))

saved_metadata = json.loads((run_dir / "metadata.json").read_text())
print({
    "policy": saved_metadata["policy"],
    "decision_batch_size": saved_metadata["decision_batch_size"],
    "batched_decision_semantics": saved_metadata["batched_decision_semantics"],
    "stop_reason": saved_metadata["stop_reason"],
    "events_recorded": saved_metadata["events_recorded"],
    "last_simulated_timestamp": transcript[-1]["timestamp"] if transcript else None,
})


## 7. Researcher-only analysis

Each component is oriented so larger means better. Survival, health and nutrition use fixed initial cohorts; reserve indices use fixed demand targets; temperature/humidity use distance from an optimum; contamination/debris are inverted. See the document for every formula and endpoint.

A mixed vector change indicates a tradeoff. It is not automatically a net improvement or decline. Immediate action effects are separated from background evolution, but do not establish long-term causal attribution.

In [ ]:
result = simulation.comparison()
print("Final vs initial:", result["classification"])
table = "| Attribute | Initial | Final | Change |\n|---|---:|---:|---:|\n"
final_vector = transcript[-1]["vector"] if transcript else simulation.baseline
for name, initial, final in zip(VECTOR_NAMES, simulation.baseline, final_vector):
    table += f"| {name} | {initial:.4f} | {final:.4f} | {final-initial:+.4f} |\n"
display(Markdown(table))

action_counts = Counter(
    event["action"].get("action", "invalid")
    if isinstance(event["action"], dict) else "invalid"
    for event in simulation.audit
)
status_counts = Counter(event["outcome"]["status"] for event in simulation.audit)
print("Action counts:", dict(action_counts))
print("Outcomes:", dict(status_counts))
usage_totals = {
    key: sum((event.get("usage") or {}).get(key, 0) for event in simulation.audit)
    for key in ("input_tokens", "output_tokens", "total_tokens")
}
print("Local prompt/completion token totals:", usage_totals)


In [ ]:
fig = plot_transcript(simulation)
plt.show()
fig.savefig(run_dir / "state_trajectories.png", dpi=150, bbox_inches="tight")

fig = plot_hive(world)
plt.show()
fig.savefig(run_dir / "final_hive.png", dpi=150, bbox_inches="tight")


### Optional explicit researcher weighting

This is a preference-weighted summary, not biological ground truth. The weights and resulting score never enter any agent prompt. Equal weights below are only an example. Change them before comparing experiments.

In [ ]:
RESEARCH_WEIGHTS = {name: 1.0 for name in VECTOR_NAMES}
weights = [RESEARCH_WEIGHTS[name] for name in VECTOR_NAMES]
print("Example weighted initial:", weighted_research_score(simulation.baseline, weights))
print("Example weighted final:  ", weighted_research_score(final_vector, weights))

# Sanity check: background plus immediate effects reconstruct overall change.
for i in range(len(VECTOR_NAMES)):
    reconstructed = sum(
        event["background_delta"][i] + event["immediate_action_delta"][i]
        for event in simulation.audit
    )
    assert abs(reconstructed - (final_vector[i] - simulation.baseline[i])) < 1e-6


## Interpretation and sources

Use multiple seeds, matched scenarios and frozen calibration to study collective behaviour. This model has coarse space/time, an incomplete brood lifecycle and assumed physiological coefficients. Training labels are synthetic; a successful local rollout is not biological validation. Batched observations deliberately share a frozen state, so action interactions are resolved by ordered physical execution and audited outcomes.

The detailed [modelling document](MODEL_DESIGN.md) maps the design sources to implementation choices:

- [TaskSelSim, §2.2](https://doi.org/10.1080/13873950701846662): task cohorts and individual food flows.
- [ApisRAM formal model, §§3–7](https://doi.org/10.2903/sp.efsa.2022.EN-7184): spatial local activities and physiological/environmental state.
- [SIMBACA, project and thesis description](https://labsticc.fr/en/teams/inuit/past-phd-thesis/phd-thomas-alves): inspectable spatial task organization.
- [Bee++, §2 and Figure 2](https://doi.org/10.3390/insects8010031): individual bees, task transitions and external foragers.
- [Qwen3.5-0.8B model](https://huggingface.co/unsloth/Qwen3.5-0.8B) and [Unsloth Qwen3.5 fine-tuning guide](https://unsloth.ai/docs/models/qwen3.5/fine-tune): local model and adapter workflow.

No equations or validation claims from the original simulators are inherited by this synthesis.
